# NB19 — Linear Regression for Time Series

> **Trend, seasonality, and lag features — linear regression applied to sequential data.**

Time series has structure that standard regression ignores:
- **Trend** — long-run direction
- **Seasonality** — regular periodic patterns
- **Autocorrelation** — past values predict future values

Linear regression handles all three if you create the right features.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)
n = 120   # 10 years of monthly data
t = np.arange(n)

# Generate synthetic time series: trend + seasonality + noise
trend      = 0.3 * t
seasonal   = 5 * np.sin(2 * np.pi * t / 12)   # annual cycle
noise      = np.random.normal(0, 1.5, n)
y          = 10 + trend + seasonal + noise

dates = pd.date_range('2014-01', periods=n, freq='ME')
ts    = pd.Series(y, index=dates)

plt.figure(figsize=(12, 4))
plt.plot(ts, linewidth=1.5, color='steelblue')
plt.title('Synthetic time series: trend + annual seasonality + noise')
plt.xlabel('Date'); plt.ylabel('Value'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# Feature engineering: trend + Fourier features for seasonality
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

def make_features(t, n_fourier=2, period=12):
    features = {'t': t}
    for k in range(1, n_fourier+1):
        features[f'sin_{k}'] = np.sin(2*np.pi*k*t/period)
        features[f'cos_{k}'] = np.cos(2*np.pi*k*t/period)
    return pd.DataFrame(features)

X_feat = make_features(t, n_fourier=2)
print("Features:"); print(X_feat.head(6))

# Train on first 96 months, test on last 24
train_idx = t < 96
X_tr, X_te = X_feat[train_idx], X_feat[~train_idx]
y_tr, y_te = y[train_idx], y[~train_idx]

m = LinearRegression().fit(X_tr, y_tr)
y_pred_tr = m.predict(X_tr)
y_pred_te = m.predict(X_te)

print(f"\nTrain R²: {r2_score(y_tr, y_pred_tr):.4f}")
print(f"Test R²:  {r2_score(y_te, y_pred_te):.4f}")

plt.figure(figsize=(12, 4))
plt.plot(dates[:96],  y[:96],        color='steelblue', linewidth=1.2, label='Train')
plt.plot(dates[96:],  y[96:],        color='orange',    linewidth=1.2, label='Test')
plt.plot(dates[:96],  y_pred_tr,     'r-', linewidth=1.5, label='Fitted')
plt.plot(dates[96:],  y_pred_te,     'g-', linewidth=2.5, label='Forecast')
plt.title('Trend + Fourier seasonality model')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# Lag features — use past values to predict future
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# AR(3) model: y_t = β₁y_{t-1} + β₂y_{t-2} + β₃y_{t-3} + trend + seasonal
def make_lag_features(y, t, lags=[1,2,3], n_fourier=2, period=12):
    df = pd.DataFrame({'t': t, 'y': y})
    for lag in lags:
        df[f'lag_{lag}'] = df['y'].shift(lag)
    for k in range(1, n_fourier+1):
        df[f'sin_{k}'] = np.sin(2*np.pi*k*t/period)
        df[f'cos_{k}'] = np.cos(2*np.pi*k*t/period)
    return df.dropna()

df_lag = make_lag_features(y, t)
feature_cols = [c for c in df_lag.columns if c != 'y']
X_lag = df_lag[feature_cols].values
y_lag = df_lag['y'].values
t_lag = df_lag['t'].values

split = t_lag < 96
X_tr2, X_te2 = X_lag[split], X_lag[~split]
y_tr2, y_te2 = y_lag[split], y_lag[~split]

m2 = LinearRegression().fit(X_tr2, y_tr2)
print(f"AR(3)+Fourier — Train R²: {m2.score(X_tr2,y_tr2):.4f}  Test R²: {m2.score(X_te2,y_te2):.4f}")
print("\nCoefficients:")
for name, coef in zip(feature_cols, m2.coef_):
    print(f"  {name:<10} {coef:.4f}")


In [ ]:
# Residual autocorrelation check
import statsmodels.api as sm
import matplotlib.pyplot as plt

resid_ts = y_tr2 - m2.predict(X_tr2)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sm.graphics.tsa.plot_acf(resid_ts, lags=20, ax=ax1)
sm.graphics.tsa.plot_pacf(resid_ts, lags=20, ax=ax2)
ax1.set_title('ACF of residuals'); ax2.set_title('PACF of residuals')
plt.tight_layout(); plt.show()
print("If ACF bars are within the blue band → residuals are white noise → model captured the structure.")


## Key Takeaways

| Technique | What it captures |
|-----------|-----------------|
| Trend feature (t) | Linear long-run direction |
| Fourier features (sin/cos) | Fixed-period seasonality |
| Lag features | Autocorrelation / momentum |
| Residual ACF | Checks if structure remains unexplained |

**Next:** NB20 — sklearn Pipelines: production-ready workflow.
